In [1]:
import FinanceDataReader as fdr
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

load_dotenv()
llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
etf_listing = fdr.StockListing("ETF/KR")
etf_listing.head()

,Symbol,Category,Name,Price,RiseFall,Change,ChangeRate,NAV,EarningRate,Volume,Amount,MarCap
0,069500,1,KODEX 200,94050,5,-660,-0.70,94029.0,33.9969,10221423,962685,213541
1,360750,4,TIGER 미국S&P500,25940,2,160,0.62,25977.0,1.3441,11532078,298601,161023
2,396500,2,TIGER 반도체TOP10,35735,5,-200,-0.56,35759.0,49.1078,8942100,319898,95198
3,133690,4,TIGER 미국나스닥100,172915,2,920,0.53,173385.0,2.7072,424825,73348,87460
4,379800,4,KODEX 미국S&P500,23705,2,145,0.62,23743.0,1.2831,20001113,473308,84852


In [2]:
name_by_ticker = dict(
    zip(etf_listing["Symbol"].astype(str).str.zfill(6), etf_listing["Name"])
)
name_by_ticker

{'069500': 'KODEX 200',
 '360750': 'TIGER 미국S&P500',
 '396500': 'TIGER 반도체TOP10',
 '133690': 'TIGER 미국나스닥100',
 '379800': 'KODEX 미국S&P500',
 '102110': 'TIGER 200',
 '459580': 'KODEX CD금리액티브(합성)',
 '488770': 'KODEX 머니마켓액티브',
 '229200': 'KODEX 코스닥150',
 '379810': 'KODEX 미국나스닥100',
 '278530': 'KODEX 200TR',
 '122630': 'KODEX 레버리지',
 '411060': 'ACE KRX금현물',
 '310970': 'TIGER MSCI Korea TR',
 '357870': 'TIGER CD금리투자KIS(합성)',
 '091160': 'KODEX 반도체',
 '498400': 'KODEX 200타겟위클리커버드콜',
 '233740': 'KODEX 코스닥150레버리지',
 '423160': 'KODEX KOFR금리액티브(합성)',
 '381180': 'TIGER 미국필라델피아반도체나스닥',
 '381170': 'TIGER 미국테크TOP10 INDXX',
 '273130': 'KODEX 종합채권(AA-이상)액티브',
 '148020': 'RISE 200',
 '360200': 'ACE 미국S&P500',
 '458730': 'TIGER 미국배당다우존스',
 '0043B0': 'TIGER 머니마켓액티브',
 '367380': 'ACE 미국나스닥100',
 '102780': 'KODEX 삼성그룹',
 '481050': 'KODEX CD1년금리플러스액티브(합성)',
 '161510': 'PLUS 고배당주',
 '455890': 'RISE 머니마켓액티브',
 '449170': 'TIGER KOFR금리액티브(합성)',
 '292150': 'TIGER 코리아TOP10',
 '395160': 'KODEX AI반도체',
 '487240': 'K

In [3]:
tickers_info = {
    "069500": {
        "category": "국내주식",
        "expense_ratio": 0.15,
        "dividend_yield": 1.8,
        "keywords": ["코스피", "대형주", "인덱스", "분산투자", "국내주식"],
    },
    "379800": {
        "category": "해외주식",
        "expense_ratio": 0.05,
        "dividend_yield": 0.0,
        "keywords": ["미국", "S&P500", "대형주", "성장", "해외주식"],
    },
    "411060": {
        "category": "배당",
        "expense_ratio": 0.01,
        "dividend_yield": 3.5,
        "keywords": ["미국", "배당", "배당성장", "저비용", "안정"],
    },
    "305540": {
        "category": "테마",
        "expense_ratio": 0.45,
        "dividend_yield": 0.0,
        "keywords": ["2차전지", "배터리", "테마", "성장", "고위험"],
    },
    "379810": {
        "category": "해외주식",
        "expense_ratio": 0.07,
        "dividend_yield": 0.0,
        "keywords": ["미국", "나스닥", "기술주", "성장", "IT"],
    },
    "381180": {
        "category": "해외주식",
        "expense_ratio": 0.49,
        "dividend_yield": 0.0,
        "keywords": ["반도체", "AI", "미국", "기술주", "고위험"],
    },
}

In [4]:
for t, info in tickers_info.items():
    name = name_by_ticker.get(t, "(이름 미확인)")
    print(f"{t} | {name:30s} | {info['category']}")

069500 | KODEX 200                      | 국내주식
379800 | KODEX 미국S&P500                 | 해외주식
411060 | ACE KRX금현물                     | 배당
305540 | TIGER 2차전지테마                   | 테마
379810 | KODEX 미국나스닥100                 | 해외주식
381180 | TIGER 미국필라델피아반도체나스닥            | 해외주식


In [5]:
from datetime import datetime, timedelta

import numpy as np


def collect_etf_price(ticker, days=365):
    now = datetime.now()
    end_date = now.strftime("%Y-%m-%d")
    start_date = (now - timedelta(days=days)).strftime("%Y-%m-%d")
    df = fdr.DataReader(ticker, start_date, end_date)
    return {"status": "real", "data": df, "ticker": ticker}


def compute_metrics(df):
    close = df["Close"].ffill()
    ret = close.pct_change().dropna()
    ann_ret = ((1 + ret.mean()) ** 252 - 1) * 100
    ann_vol = ret.std() * np.sqrt(252) * 100
    cum = (1 + ret).cumprod()
    mdd = ((cum - cum.cummax()) / cum.cummax()).min() * 100
    return {
        "return_1y": round(ann_ret, 2),
        "volatility": round(ann_vol, 2),
        "mdd": round(mdd, 2),
    }

In [6]:
def risk_from_vol(vol):
    if vol < 5:
        return "낮음"
    elif vol < 15:
        return "약간 낮음"
    elif vol < 20:
        return "보통"
    elif vol < 25:
        return "약간 높음"
    else:
        return "높음"


etf_data = []
for t, info in tickers_info.items():
    result = collect_etf_price(t)
    metrics = compute_metrics(result["data"])
    name = name_by_ticker.get(t, f"ETF_{t}")
    data = {
        "ticker": t,
        "name": name,
        "risk_level": risk_from_vol(metrics["volatility"]),
        "data_status": result["status"],
        **info,
        **metrics,
    }
    etf_data.append(data)

etf_data[:2]

[{'ticker': '069500',
  'name': 'KODEX 200',
  'risk_level': '높음',
  'data_status': 'real',
  'category': '국내주식',
  'expense_ratio': 0.15,
  'dividend_yield': 1.8,
  'keywords': ['코스피', '대형주', '인덱스', '분산투자', '국내주식'],
  'return_1y': np.float64(221.8),
  'volatility': np.float64(36.6),
  'mdd': np.float64(-20.62)},
 {'ticker': '379800',
  'name': 'KODEX 미국S&P500',
  'risk_level': '약간 낮음',
  'data_status': 'real',
  'category': '해외주식',
  'expense_ratio': 0.05,
  'dividend_yield': 0.0,
  'keywords': ['미국', 'S&P500', '대형주', '성장', '해외주식'],
  'return_1y': np.float64(42.41),
  'volatility': np.float64(13.32),
  'mdd': np.float64(-5.7)}]

In [7]:
from langchain_core.documents import Document


def get_etf_context(etf):
    return f"""
    - 이름: {etf["name"]}
    - 카테고리: {etf["category"]}
    - 키워드: {", ".join(etf["keywords"])}
    - 수수료: {etf["expense_ratio"]}%
    - 배당 수익률: {etf["dividend_yield"]}%
    - 연율 수익률: {etf["return_1y"]}%
    - 연율 변동성: {etf["volatility"]}%
    - 최대 하락: {etf["mdd"]}%
    """


def generate_description(etf):
    prompt = (
        "다음 ETF의 특징을 한국어 1~2문장으로 간결히 설명하세요.\n"
        + get_etf_context(etf)
    )

    return str(llm.invoke(prompt).content).strip()


documents = []
for etf in etf_data:
    description = generate_description(etf)
    page_content = f"{description}\n" + get_etf_context(etf)
    metadata = {k: v for k, v in etf.items() if k != "keywords"}
    metadata["keywords"] = ", ".join(etf["keywords"])
    documents.append(Document(page_content=page_content, metadata=metadata))


In [8]:
from langchain_community.vectorstores import FAISS

vs = FAISS.from_documents(documents, embeddings)
vs.index.ntotal

6

In [9]:
query = "미국 기술주에 투자하고 싶어요."
vs.similarity_search_with_score(query, k=3)

[(Document(id='1015ba68-6704-4eaa-bfec-60f02feeab3e', metadata={'ticker': '381180', 'name': 'TIGER 미국필라델피아반도체나스닥', 'risk_level': '높음', 'data_status': 'real', 'category': '해외주식', 'expense_ratio': 0.49, 'dividend_yield': 0.0, 'return_1y': np.float64(169.62), 'volatility': np.float64(31.34), 'mdd': np.float64(-10.35), 'keywords': '반도체, AI, 미국, 기술주, 고위험'}, page_content='TIGER 미국필라델피아반도체나스닥 ETF는 미국의 반도체 및 기술주에 투자하는 고위험 상품으로, 최근 연율 수익률이 169.62%에 달하지만 배당 수익률은 0.0%입니다. 수수료는 0.49%이며, 연율 변동성이 31.34%로 높은 변동성을 보입니다.\n\n    - 이름: TIGER 미국필라델피아반도체나스닥\n    - 카테고리: 해외주식\n    - 키워드: 반도체, AI, 미국, 기술주, 고위험\n    - 수수료: 0.49%\n    - 배당 수익률: 0.0%\n    - 연율 수익률: 169.62%\n    - 연율 변동성: 31.34%\n    - 최대 하락: -10.35%\n    '),
  np.float32(1.2223653)),
 (Document(id='6df7d888-b9f6-4c54-958d-5b91e9b634c1', metadata={'ticker': '379810', 'name': 'KODEX 미국나스닥100', 'risk_level': '보통', 'data_status': 'real', 'category': '해외주식', 'expense_ratio': 0.07, 'dividend_yield': 0.0, 'return_1y': np.float64(53.33), 'volatility': 

In [10]:
etf_knowledge_base = [
    {
        "ticker": "069500",
        "name": "KODEX 200",
        "category": "국내주식",
        "description": "KOSPI 200 지수 추적. 수수료 0.15%. 대형주 중심 분산투자.",
        "risk": "중간",
        "expense_ratio": 0.15,
    },
    {
        "ticker": "379800",
        "name": "KODEX 미국S&P500TR",
        "category": "해외주식",
        "description": "S&P500 추적, 배당 자동 재투자. 수수료 0.05%.",
        "risk": "중간",
        "expense_ratio": 0.05,
    },
    {
        "ticker": "461460",
        "name": "KODEX 미국나스닥100TR",
        "category": "해외주식",
        "description": "나스닥100 기술주 중심. 높은 성장성/변동성. 수수료 0.05%.",
        "risk": "높음",
        "expense_ratio": 0.05,
    },
]

etf_names = [f"{e['name']}: {e['description']}" for e in etf_knowledge_base]
etf_names

['KODEX 200: KOSPI 200 지수 추적. 수수료 0.15%. 대형주 중심 분산투자.',
 'KODEX 미국S&P500TR: S&P500 추적, 배당 자동 재투자. 수수료 0.05%.',
 'KODEX 미국나스닥100TR: 나스닥100 기술주 중심. 높은 성장성/변동성. 수수료 0.05%.']

In [11]:
from ast import literal_eval

prompt = f"""
다음 ETF 데이터를 보고 실제 투자자가 물어볼 법한 질문 5개를 생성하세요.

[ETF 목록]
{str(etf_names)}

[형식]
["질문1", "질문2", "질문3", "질문4", "질문5"]
"""

synthetic = llm.invoke(prompt).content
queries = literal_eval(str(synthetic))
queries

['KODEX 200 ETF의 수수료가 다른 ETF에 비해 높은 편인가요?',
 'KODEX 미국S&P500TR ETF의 배당금은 어떻게 재투자되나요?',
 'KODEX 미국나스닥100TR ETF의 변동성이 크다고 하는데, 어떤 리스크가 있나요?',
 'KODEX 200 ETF와 KODEX 미국S&P500TR ETF의 성과를 비교할 때 어떤 점을 고려해야 하나요?',
 'KODEX 미국나스닥100TR ETF에 투자할 때 장기 투자와 단기 투자 중 어떤 전략이 더 효과적인가요?']

In [12]:
from kiwipiepy import Kiwi

kiwi = Kiwi()


def kiwi_tokenize(text):
    tokens = kiwi.tokenize(text)
    result = []
    for token in tokens:
        if token.tag in ("SL", "SN", "NNG", "NNP"):
            result.append(token.form.lower())
    return result


corpus = [kiwi_tokenize(doc.page_content) for doc in documents]

In [13]:
from rank_bm25 import BM25Okapi

bm25 = BM25Okapi(corpus)


def bm25_search(query, k=3):
    tokens = kiwi_tokenize(query)
    scores = bm25.get_scores(tokens)
    top_idx = np.argsort(scores)[::-1][:k]
    return [(documents[i], scores[i]) for i in top_idx if scores[i] > 0]


tokens = kiwi_tokenize("KODEX 200 ETF는 어떤 종류의 주식에 투자하나요?")

In [ ]:
for q in queries:
    v_result = vs.similarity_search_with_score(q, k=1)
    b_result = bm25_search(q, k=1)
    v_name = v_result[0][0].metadata["name"] if v_result else "-"
    b_name = b_result[0][0].metadata["name"] if b_result else "-"
    print(f"{q:20s} | {v_name:20s} | {b_name:20s}")

KODEX 200 ETF의 수수료가 다른 ETF에 비해 높은 편인가요? | KODEX 200            | KODEX 200           
KODEX 미국S&P500TR ETF는 배당금이 얼마나 자주 지급되나요? | KODEX 미국S&P500       | KODEX 미국S&P500      
KODEX 미국나스닥100TR ETF의 변동성이 크다고 하는데, 어떤 리스크가 있을까요? | KODEX 미국나스닥100       | KODEX 미국나스닥100      
KODEX 200 ETF와 KODEX 미국S&P500TR ETF 중 어떤 것이 더 안정적인 투자처인가요? | KODEX 미국S&P500       | KODEX 미국S&P500      
KODEX 미국나스닥100TR ETF에 투자할 때 주의해야 할 점은 무엇인가요? | KODEX 미국나스닥100       | KODEX 미국나스닥100      


In [ ]:
query = queries[0]
tokens = kiwi_tokenize(query)
params = [(0.5, 0.25), (1.5, 0.75), (2.5, 0.9)]
for k1, b in params:
    bm25_temp = BM25Okapi(corpus, k1=k1, b=b)
    scores = bm25_temp.get_scores(tokens)
    top3 = np.argsort(scores)[::-1][:3]
    print(f"k1={k1}, b={b}")
    for i in top3:
        print(f"  - {documents[i].metadata['name']}({scores[i]:.2f})")
    print()

k1=0.5, b=0.25
  - KODEX 200(2.03)
  - TIGER 2차전지테마(0.47)
  - KODEX 미국S&P500(0.46)

k1=1.5, b=0.75
  - KODEX 200(2.45)
  - TIGER 2차전지테마(0.57)
  - KODEX 미국S&P500(0.53)

k1=2.5, b=0.9
  - KODEX 200(2.68)
  - TIGER 2차전지테마(0.62)
  - KODEX 미국S&P500(0.57)



In [ ]:
def filtered_search(vs, query, filters=None, k=5, fetch_k=20):
    results = vs.similarity_search_with_score(query, k=fetch_k)
    if not filters:
        return results[:k]
    filtered = []
    for doc, score in results:
        for key, condition in filters.items():
            val = doc.metadata.get(key)
            if isinstance(condition, dict):
                for op, val in condition.items():
                    if op == "less_than":
                        match = val > score
                    elif op == "greater_than":
                        match = val < score
            else:
                match = val == condition
        if match:
            filtered.append((doc, score))
    return filtered[:k]


In [ ]:
results = filtered_search(
    vs,
    "저비용 해외 ETF",
    filters={"category": "해외주식", "expense_ratio": {"gt": 0.05}},
    k=3,
)
for r in results:
    print(r)

(Document(id='0008005c-e720-4635-a4a6-b5e3a7498ed5', metadata={'ticker': '381180', 'name': 'TIGER 미국필라델피아반도체나스닥', 'risk_level': '높음', 'data_status': 'real', 'category': '해외주식', 'expense_ratio': 0.49, 'dividend_yield': 0.0, 'return_1y': np.float64(171.26), 'volatility': np.float64(31.35), 'mdd': np.float64(-10.35), 'keywords': '반도체, AI, 미국, 기술주, 고위험'}, page_content='TIGER 미국필라델피아반도체나스닥 ETF는 미국의 반도체 및 기술주에 투자하는 고위험 상품으로, 최근 1년간 연율 수익률이 171.26%에 달하지만, 변동성이 31.35%로 높아 주의가 필요합니다. 수수료는 0.49%이며, 배당 수익률은 0.0%입니다.\n\n    - 이름: TIGER 미국필라델피아반도체나스닥\n    - 카테고리: 해외주식\n    - 키워드: 반도체, AI, 미국, 기술주, 고위험\n    - 수수료: 0.49%\n    - 배당 수익률: 0.0%\n    - 연율 수익률: 171.26%\n    - 연율 변동성: 31.35%\n    - 최대 하락: -10.35%\n    '), np.float32(1.0596588))
(Document(id='443e882d-fe43-4776-90b7-675e0c00c1c5', metadata={'ticker': '379800', 'name': 'KODEX 미국S&P500', 'risk_level': '약간 낮음', 'data_status': 'real', 'category': '해외주식', 'expense_ratio': 0.05, 'dividend_yield': 0.0, 'return_1y': np.float64(40.68), 'volatility': 

In [ ]:
import json


def smart_filtered_search(query):
    prompt = f"""사용자 쿼리에서 ETF 검색 필터를 추출하세요.

    쿼리 : {query}

    사용 가능한 필터 필드:
    - category : 국내 주식, 해외 주식, 배당, 테마 ...
    - risk_level : 낮음, 중간, 높음
    - expense_ratio : 숫자(% "less_than"/"greater_than")
    - dividend_yield : 숫자 (%)
    - return_1y : 숫자 (%)
    - volatility: 숫자 (%)

    코드 블럭없이 JSON 형식으로만 응답하세요. 필터가 없으면 {{}}로 답변하세요.
    """
    response = llm.invoke([{"role": "user", "content": prompt}])
    return json.loads(str(response.content))


test_query = "수수료 0.1% 이하이면서 배당 3% 이상인 ETF"
filters = smart_filtered_search(test_query)
filters


{'expense_ratio': {'less_than': 0.1}, 'dividend_yield': {'greater_than': 3}}

In [ ]:
def smart_document_search(query, k=3):
    filters = smart_filtered_search(query)
    results = filtered_search(vs, query, filters=filters, k=k)
    if not results:
        print("검색 결과가 없습니다. 조건 없이 재검색...")
        results = vs.similarity_search_with_score(query, k=k)
    return results